# Basic Test

In [1]:
import torch

from llm_circuits.circuits.replacement_model import compare_models
from llm_circuits.models.qwen3 import load_qwen3
from llm_circuits.transcoders.circuit_tracer_loader import load_transcoder

model, tokenizer = load_qwen3("0.6b")
model.eval()
print(f"Model: {type(model).__name__}, device: {model.device}")

loaded = load_transcoder("0.6b", device=model.device, lazy_decoder=True)
tc = loaded.transcoder
print(f"Transcoder: {type(loaded.transcoder).__name__}")
print(f"Repo: {loaded.repo_id}")
print(f"Config keys: {list(loaded.config.keys())}")

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
tokens = [tokenizer.decode(t) for t in input_ids[0]]
print(f"\nPrompt: {prompt!r}")
print(f"Tokens: {tokens}\n")

[02/13/26 00:48:22] INFO     Loading model Qwen/Qwen3-0.6B (dtype=torch.bfloat16, device_map=auto,                 
                             cache_dir=/Users/wdk0082/Projects/llm-circuits/.cache/models)

`torch_dtype` is deprecated! Use `dtype` instead!


Model: Qwen3ForCausalLM, device: mps:0


[02/13/26 00:48:25] INFO     Loading transcoders from cache mwhanna/qwen3-0.6b-transcoders-lowl0                   
                             (cache_dir=/Users/wdk0082/Projects/llm-circuits/.cache/transcoders)

Transcoder: TranscoderSet
Repo: mwhanna/qwen3-0.6b-transcoders-lowl0
Config keys: ['feature_input_hook', 'feature_output_hook', 'model_kind', 'model_name', 'revision', 'scan']

Prompt: 'The capital of France is'
Tokens: ['The', ' capital', ' of', ' France', ' is']



In [2]:
with torch.no_grad():
    result = compare_models(
        model, tc, input_ids,
    )

In [3]:
# --- Print per-position metrics -------------------------------------------
orig_preds = result.original_logits.argmax(dim=-1)
repl_preds = result.replacement_logits.argmax(dim=-1)

print(
    f"{'Pos':>3}  {'Token':>12}  {'Orig pred':>12}  {'Repl pred':>12}"
    f"  {'KL div':>10}  {'Cos sim':>10}  {'Top-1':>6}"
)
print("-" * 78)
for i, tok in enumerate(tokens):
    orig_tok = tokenizer.decode(orig_preds[i].item())
    repl_tok = tokenizer.decode(repl_preds[i].item())
    kl = result.kl_divergence[i].item()
    cos = result.cosine_similarity[i].item()
    agree = "yes" if result.top1_agreement[i].item() else "NO"
    print(
        f"{i:3d}  {tok:>12s}  {orig_tok:>12s}  {repl_tok:>12s}"
        f"  {kl:10.4f}  {cos:10.4f}  {agree:>6s}"
    )

# --- Summary --------------------------------------------------------------
mean_kl = result.kl_divergence.mean().item()
mean_cos = result.cosine_similarity.mean().item()
pct_agree = result.top1_agreement.float().mean().item() * 100

print(f"\nMean KL divergence:   {mean_kl:.4f}")
print(f"Mean cosine sim:      {mean_cos:.4f}")
print(f"Top-1 agreement:      {pct_agree:.1f}%")

# --- Per-layer reconstruction error ---------------------------------------
if result.reconstruction_errors:
    print(f"\n{'Layer':>5}  {'Mean L2 error':>14}")
    print("-" * 22)
    for layer_idx in sorted(result.reconstruction_errors):
        err = result.reconstruction_errors[layer_idx].mean().item()
        print(f"{layer_idx:5d}  {err:14.4f}")

Pos         Token     Orig pred     Repl pred      KL div     Cos sim   Top-1
------------------------------------------------------------------------------
  0           The      Question           and     11.3237      0.3322      NO
  1       capital            of           and      9.1288      0.4550      NO
  2            of           the           and      5.2948      0.4968      NO
  3        France            is           and      4.5173      0.5453      NO
  4            is         Paris           and     11.9980      0.6182      NO

Mean KL divergence:   8.4525
Mean cosine sim:      0.4895
Top-1 agreement:      0.0%

Layer   Mean L2 error
----------------------
    0          2.2342
    1          2.6411
    2          3.3501
    3          7.0030
    4          7.4859
    5         10.0059
    6         10.6377
    7         12.5757
    8         14.2794
    9         15.4224
   10         19.6219
   11         24.3343
   12         26.1386
   13         24.3173
   14        

In [4]:
output_ids_org = torch.argmax(result.original_logits, axis=1)
text = tokenizer.decode(output_ids_org)
print(output_ids_org, text)

output_ids_rc = torch.argmax(result.replacement_logits, axis=1)
text_rc = tokenizer.decode(output_ids_rc)
print(output_ids_rc, text_rc)

tensor([15846,   315,   279,   374, 12095], device='mps:0')  Question of the is Paris
tensor([323, 323, 323, 323, 323], device='mps:0')  and and and and and


In [14]:
dir(result)
print(result.original_activations[20])
print(result.replacement_activations[20])

tensor([[-8.3984e-02, -9.8633e-02,  2.4023e-01,  ..., -5.9814e-02,
          2.5177e-03,  1.1841e-02],
        [ 2.9062e+00, -1.4062e+00,  5.0938e+00,  ...,  2.9492e-01,
          2.0312e+00, -1.5078e+00],
        [-1.3574e-01, -3.3750e+00,  9.3125e+00,  ..., -1.2188e+00,
          2.2031e+00, -8.2812e-01],
        [-6.8970e-03, -1.1953e+00,  5.2812e+00,  ..., -6.7969e-01,
          1.8281e+00,  2.6250e+00],
        [-1.4375e+00, -1.2344e+00,  1.0750e+01,  ..., -1.7656e+00,
          1.4766e+00, -1.1094e+00]], device='mps:0', dtype=torch.bfloat16)
tensor([[-1.0689, -1.5791,  3.3515,  ...,  0.6752,  0.1309, -0.0317],
        [-1.0897, -1.6263,  3.2386,  ...,  0.7777,  0.2023,  0.1469],
        [-1.0806, -1.6446,  3.2875,  ...,  0.7752,  0.0960,  0.1511],
        [-1.0829, -1.6411,  3.2565,  ...,  0.7732,  0.1494,  0.1423],
        [-1.0602, -1.6933,  3.2693,  ...,  0.7516,  0.0389,  0.1074]],
       device='mps:0')
